# **LightGBM Model Development and Evaluation Script**

---

## Modeling Pipeline

- **Baseline Model Development:** A standard model was trained using default/reasonable hyperparameters to establish a performance benchmark.

- **Hyperparameter Optimization:** Each baseline model was optimized using two complementary techniques:
  - ***`Optuna`***: Efficient, adaptive search with pruning for rapid convergence to high-performing configurations.
  - ***`RandomizedSearchCV`***: Sklearn-compatible baseline optimization for fair comparison and reproducibility.

- **Cross-Validation and Robustness Assessment:** Each model variant was evaluated using ***`TimeSeriesSplit`*** to preserve temporal order and prevent look-ahead bias. Metrics were aggregated across folds to assess stability.

- **Overfitting Analysis:** A detailed comparison between cross-validation metrics and test set results was conducted. Additional metrics, including ***`RMSE ratio`*** and ***`R² gap`***, were computed to quantify overfitting and assess model generalization. ***`Directional accuracy`*** and ***`financial metrics`*** (Sharpe Ratio, Max Drawdown) were also calculated for trading-relevant evaluation.

---

## Persisted Artifacts

To ensure reproducibility, transparency, and extendability, the following artifacts have been saved for **each model**:

- **Optimized Model Performance:** Individual CSV files capturing the performance of each model variant:
    - ***LightGBM (Baseline)***
    - ***LightGBM (Optuna)***
    - ***LightGBM (RandomizedSearchCV)***

- **Best Variation Performance:** A CSV file containing only the metrics of the best-performing variation per model.

- **Summary of Model Performance:** A consolidated, extendable CSV file (`AllModel_OverallPerformance.csv`) including:
    - Cross-validation results (`CV MSE`, `CV MAE`, `CV RMSE`, `CV R²`, `CV MAPE`)
    - Test set results (`Test MSE`, `Test MAE`, `Test RMSE`, `Test R²`, `Test MAPE`)
    - Financial metrics (`Sharpe Ratio`, `Sortino Ratio`, `Max Drawdown`, `Directional Accuracy`)
    - Overfitting metrics (`R² gap`, `RMSE ratio`)
    - Overfitting status and model generalization label

- **Overfitting DataFrame:** An extendable CSV (`AllModel_OverfittingAnalysis.csv`) capturing overfitting analysis metrics across all models and variations.

- **Best Model per Algorithm:** The serialized best-performing variant of each algorithm for ensemble consideration or deployment.

- **Model Comparison:** A summary notebook or script that loads `AllModel_OverallPerformance.csv` and generates publication-ready comparison visualizations.

Together, these artifacts provide a complete, reproducible record of the modeling process, facilitating model tracking, comparison, selection, and deployment.

## Import Libraries and Root Configuration

In [1]:
""" Configure the utilities module path for imports """
import sys
import os
from pathlib import Path

# get project root as parent of current working directory
PROJECT_ROOT = Path(os.getcwd()).parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
# artifacts root
DATA_ROOT = PROJECT_ROOT / "data"
FEATURE_ROOT = PROJECT_ROOT / "artifacts" / "FeatureSelection"
FIGURE_ROOT = PROJECT_ROOT / "visualizations" / "ModelEvaluation"
MODEL_ROOT = PROJECT_ROOT / "artifacts" / "Models"
PERFORMANCE_ROOT = PROJECT_ROOT / "artifacts" / "ModelPerformance"

In [3]:
# records and calculations
import pandas as pd
import numpy as np

# avoid minor warnings
import warnings
warnings.filterwarnings('ignore')

# visualizations
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from scipy.stats import uniform, randint, loguniform

# gradient boosting model
import lightgbm as lgb

# optimization
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# import helper functions
from src.utilities import Evaluator, DataHandler, ModelPersister

## Load Dataset and Artifacts

In [4]:
df, x, y = DataHandler.load_dataset(DATA_ROOT / "gold_price_engineered.csv", target_col="target")
artifacts = DataHandler.load_artifacts(FEATURE_ROOT, cv_method='tscv')

In [5]:
# check dataset loading
df.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag2,EURUSD_Return_lag3,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5,target
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,-0.002650,0.023711,-0.004229,-0.005142,...,0.023711,-0.004229,-0.005142,0.008367,-0.002650,0.023711,-0.004229,-0.005142,0.008367,0.003739
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.019642,-0.002650,0.023711,-0.004229,...,-0.002650,0.023711,-0.004229,-0.005142,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.010838
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,0.003739,0.019642,-0.002650,0.023711,...,0.019642,-0.002650,0.023711,-0.004229,0.003739,0.019642,-0.002650,0.023711,-0.004229,-0.017311
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010838,0.003739,0.019642,-0.002650,...,0.003739,0.019642,-0.002650,0.023711,0.010838,0.003739,0.019642,-0.002650,0.023711,-0.014661
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.017311,0.010838,0.003739,0.019642,...,0.010838,0.003739,0.019642,-0.002650,-0.017311,0.010838,0.003739,0.019642,-0.002650,-0.002307


In [6]:
# check input features
x.head()

,Date,SPX_Return,GLD_Return,USO_Return,SLV_Return,EURUSD_Return,SPX_Return_lag1,SPX_Return_lag2,SPX_Return_lag3,SPX_Return_lag4,...,EURUSD_Return_lag1,EURUSD_Return_lag2,EURUSD_Return_lag3,EURUSD_Return_lag4,EURUSD_Return_lag5,GLD_Return_lag1,GLD_Return_lag2,GLD_Return_lag3,GLD_Return_lag4,GLD_Return_lag5
0,2008-01-10,0.007948,0.019642,-0.016346,0.034858,0.009339,-0.002650,0.023711,-0.004229,-0.005142,...,-0.002650,0.023711,-0.004229,-0.005142,0.008367,-0.002650,0.023711,-0.004229,-0.005142,0.008367
1,2008-01-11,-0.013595,0.003739,-0.012564,0.000996,-0.000739,0.019642,-0.002650,0.023711,-0.004229,...,0.019642,-0.002650,0.023711,-0.004229,-0.005142,0.019642,-0.002650,0.023711,-0.004229,-0.005142
2,2008-01-14,0.010871,0.010838,0.015871,0.012627,0.005337,0.003739,0.019642,-0.002650,0.023711,...,0.003739,0.019642,-0.002650,0.023711,-0.004229,0.003739,0.019642,-0.002650,0.023711,-0.004229
3,2008-01-15,-0.024925,-0.017311,-0.019798,-0.027396,-0.004499,0.010838,0.003739,0.019642,-0.002650,...,0.010838,0.003739,0.019642,-0.002650,0.023711,0.010838,0.003739,0.019642,-0.002650,0.023711
4,2008-01-16,-0.005612,-0.014661,-0.012778,-0.011368,-0.009326,-0.017311,0.010838,0.003739,0.019642,...,-0.017311,0.010838,0.003739,0.019642,-0.002650,-0.017311,0.010838,0.003739,0.019642,-0.002650


In [7]:
# check target feature
y.head()

0    0.003739
1    0.010838
2   -0.017311
3   -0.014661
4   -0.002307
Name: target, dtype: float64

In [8]:
# load train/test split data
x_train, x_test, y_train, y_test = artifacts['x_train'], artifacts['x_test'], artifacts['y_train'], artifacts['y_test']
cv = artifacts['cv']

# **Baseline Modeling**

In [9]:
# model config
BASELINE_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'n_estimators': 100,
    'max_depth': 6,
    'num_leaves': 31,
    'subsample': 1.0,
    'colsample_bytree': 1.0,
    'random_state': 42,
    'verbosity': -1
}

# train baseline model
baseline_model = lgb.LGBMRegressor(**BASELINE_PARAMS)
baseline_model.fit(x_train, y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


##  Apply Model to Make Prediction

In [10]:
# baseline prediction on train and test set
y_train_pred_baseline = baseline_model.predict(x_train)
y_test_pred_baseline = baseline_model.predict(x_test)

## Evaluating Model Performance
Calculating Errors for train and test data

In [11]:
# evaluation of train metrics
train_metrics_baseline = Evaluator.calculate_metrics(y_train, y_train_pred_baseline)

# evaluation of test metrics
test_metrics_baseline = Evaluator.calculate_metrics(y_test, y_test_pred_baseline)

In [12]:
# calculate directional time-series specific accuracy
train_acc_baseline = Evaluator.directional_accuracy(y_train, y_train_pred_baseline)
test_acc_baseline = Evaluator.directional_accuracy(y_test, y_test_pred_baseline)

In [13]:
# calcluate financial metrics of test data for baseline model
fin_baseline = Evaluator.financial_metrics('LightGBM (Baseline)', y_test, y_test_pred_baseline)

display(fin_baseline)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,LightGBM (Baseline),-0.8891,-1.3673,-28.1391,-16.9772


In [14]:
# baseline model performance
baseline_perf = Evaluator.performance_table(train_metrics_baseline + [train_acc_baseline], test_metrics_baseline + [test_acc_baseline])

print("LightGBM - Baseline Modeling Performance")
display(baseline_perf)

LightGBM - Baseline Modeling Performance


,Metrics,Training,Test
0,MSE,0.0001,0.0001
1,MAE,0.0078,0.0061
2,RMSE,0.0110,0.0084
3,R2 Score,0.3747,-0.0594
4,MAPE,5661.8160,48661.6782
5,Directional Accuracy (%),68.0175,50.7659


# **Optuna Hyperparameter Optimzation**

## Find Best Parameters

In [15]:
# define objective function for hyperparameter optimization
def objective(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 125),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "random_state": 42,
        "verbosity": -1
    }

    # create model
    model = lgb.LGBMRegressor(**params)

    # fit model
    model.fit(x_train, y_train, eval_set=[(x_train, y_train)], eval_metric='rmse')

    # predict and evaluate on train; optuna will rank by this
    y_pred = model.predict(x_train)
    rmse = np.sqrt(mean_squared_error(y_train, y_pred))

    return rmse

In [16]:
# enable pruning to skip the bad trials early
pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=20)

# create study object
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

# optimize hyperparameters
study.optimize(objective, n_trials=50, show_progress_bar=True)

# best parameter from optuna
best_params_optuna = study.best_params

Best trial: 44. Best value: 1.2756e-05: 100%|██████████| 50/50 [04:14<00:00,  5.09s/it] 


## Train optimized model and Apply Model to Make Predictions

In [17]:
# train optimized model
optuna_model = lgb.LGBMRegressor(**best_params_optuna, random_state=42, verbosity=-1)
optuna_model.fit(x_train, y_train)

,boosting_type,'gbdt'
,num_leaves,120
,max_depth,9
,learning_rate,0.1525800780325201
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [18]:
# apply model to make predictions on train and test set
y_train_pred_optuna = optuna_model.predict(x_train)
y_test_pred_optuna = optuna_model.predict(x_test)

## Evaluating Optimized Model Performance
Calculating Errors for train and test data

In [ ]:
""" calculate train and test metrics
return [mse, mae, rmse, r2, mape] """
train_metrics_optuna = Evaluator.calculate_metrics(y_train, y_train_pred_optuna)
test_metrics_optuna = Evaluator.calculate_metrics(y_test, y_test_pred_optuna)

In [ ]:
# calculate directional time-series specific accuracy
train_acc_optuna = Evaluator.directional_accuracy(y_train, y_train_pred_optuna)
test_acc_optuna = Evaluator.directional_accuracy(y_test, y_test_pred_optuna)

In [ ]:
# calcluate financial metrics of test data for Optuna model
fin_optuna = Evaluator.financial_metrics('XGBoost (Optuna)', y_test, y_test_pred_optuna)

display(fin_optuna)

,Model,Sharpe Ratio,Sortino Ratio,Max Drawdown,Total Return (%)
0,XGBoost (Optuna),0.0806,0.1311,-17.6019,4.08


In [ ]:
# performance table of optuna optimization
optuna_perf = Evaluator.performance_table(train_metrics_optuna + [train_acc_optuna], test_metrics_optuna + [test_acc_optuna])

print("XGBoost - optuna Modeling Performance")
display(optuna_perf)

XGBoost - optuna Modeling Performance


,Metrics,Training,Test
0,MSE,0.0002,0.0001
1,MAE,0.0095,0.0060
2,RMSE,0.0139,0.0081
3,R2 Score,-0.0000,-0.0001
4,MAPE,4614.0165,9118.2591
5,Directional Accuracy (%),52.1906,51.2035
